# LeetCode 2006 — Count Number of Pairs With Absolute Difference K bằng Python

**Link đề:** https://leetcode.com/problems/count-number-of-pairs-with-absolute-difference-k/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Counting

```python
class Solution:
    def countKDifference(self, nums: List[int], k: int) -> int:

```

## Mục tiêu học
1. Hiểu đề: đếm cặp **chỉ số** `i < j` với `|nums[i] − nums[j]| == k`.
2. Nắm `abs()`, vòng lặp lồng nhau `j` bắt đầu từ `i + 1`, `dict`/`Counter`, `dict.get(key, 0)`.
3. Viết vét cạn `O(n²)` (hint của đề) rồi tối ưu bằng **hash map đếm số đã gặp** — "phần còn thiếu" là `x − k` và `x + k`.
4. Học cách thứ ba: **đếm theo giá trị** `cnt[v] * cnt[v + k]`, và cách mảng đếm khi giá trị nhỏ.
5. Trace, chứng minh không đếm trùng, so sánh 4 cách, test đối chiếu, bài tập biến thể.

## 1. Đề bài yêu cầu gì?

Cho mảng `nums` và số nguyên `k`. Trả về số cặp `(i, j)` với `i < j` sao cho `|nums[i] − nums[j]| == k`.

`|x|` (giá trị tuyệt đối) = `x` nếu `x ≥ 0`, `−x` nếu `x < 0`.

| `nums` | `k` | Kết quả | Các cặp chỉ số |
|---|:-:|:-:|---|
| `[1,2,2,1]` | 1 | `4` | (0,1), (0,2), (1,3), (2,3) |
| `[1,3]` | 3 | `0` | không có |
| `[3,2,1,5,4]` | 2 | `3` | (0,2): 3−1; (0,3): 5−3; (1,4): 4−2 |

**Constraints:** `1 <= nums.length <= 200`, `1 <= nums[i] <= 100`, `1 <= k <= 99` → `k ≥ 1` (không bao giờ là 0).

Chú ý ví dụ 1: chỉ có hai **giá trị** (1 và 2) nhưng **4 cặp chỉ số** — đếm theo vị trí.

In [1]:
nums, k = [1, 2, 2, 1], 1
for i in range(len(nums)):
    for j in range(i + 1, len(nums)):
        d = abs(nums[i] - nums[j])
        print(f"(i={i}, j={j}): |{nums[i]} - {nums[j]}| = {d}", "<- đếm" if d == k else "")

(i=0, j=1): |1 - 2| = 1 <- đếm
(i=0, j=2): |1 - 2| = 1 <- đếm
(i=0, j=3): |1 - 1| = 0 
(i=1, j=2): |2 - 2| = 0 
(i=1, j=3): |2 - 1| = 1 <- đếm
(i=2, j=3): |2 - 1| = 1 <- đếm


## 2. Nền tảng Python cần biết

### 2.1. `abs()` — giá trị tuyệt đối

In [2]:
print(abs(3 - 1), abs(1 - 3))   # 2 2 — thứ tự trừ không quan trọng
print(abs(-7), abs(0))

2 2
7 0


### 2.2. Hai vòng lặp với `j` bắt đầu từ `i + 1`

`for j in range(i + 1, n)` đảm bảo `i < j`: mỗi cặp được xét **đúng một lần**, không xét cặp `(i, i)`. Số cặp là `n(n−1)/2`.

In [3]:
n = 4
pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
print(pairs)
print("số cặp:", len(pairs), "= n(n-1)/2 =", n * (n - 1) // 2)

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
số cặp: 6 = n(n-1)/2 = 6


### 2.3. `dict` làm bộ đếm, `dict.get(key, 0)`

`seen = {}` lưu `giá trị -> số lần đã gặp`. `seen.get(x, 0)` trả số lần của `x`, hoặc `0` nếu chưa có — tránh `KeyError`.

In [4]:
seen = {}
for x in [1, 2, 2]:
    seen[x] = seen.get(x, 0) + 1
    print("gặp", x, "->", seen)
print("số lần gặp 5:", seen.get(5, 0))

gặp 1 -> {1: 1}
gặp 2 -> {1: 1, 2: 1}
gặp 2 -> {1: 1, 2: 2}
số lần gặp 5: 0


### 2.4. `collections.Counter`

`Counter(nums)` đếm số lần xuất hiện của mỗi giá trị trong một dòng. Truy cập key chưa có trả `0`. Thuộc tính `.items()` cho các cặp `(giá trị, số lần)`.

In [5]:
from collections import Counter
cnt = Counter([3, 2, 1, 5, 4, 3])
print(cnt)
print(cnt[3], cnt[100])
for v, c in sorted(cnt.items()):
    print("giá trị", v, "xuất hiện", c, "lần")

Counter({3: 2, 2: 1, 1: 1, 5: 1, 4: 1})
2 0
giá trị 1 xuất hiện 1 lần
giá trị 2 xuất hiện 1 lần
giá trị 3 xuất hiện 2 lần
giá trị 4 xuất hiện 1 lần
giá trị 5 xuất hiện 1 lần


## 3. Cách 1 — Vét cạn mọi cặp `O(n²)`

Đúng như hint của đề: *"Can we use a nested for loop?"*

In [6]:
def count_k_diff_brute(nums, k):
    n = len(nums)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if abs(nums[i] - nums[j]) == k:
                count += 1
    return count

print(count_k_diff_brute([1, 2, 2, 1], 1))     # 4
print(count_k_diff_brute([1, 3], 3))           # 0
print(count_k_diff_brute([3, 2, 1, 5, 4], 2))  # 3

4
0
3


### Code nộp LeetCode — Cách 1 (Accept vì n ≤ 200)

In [7]:
from typing import List

class SolutionBrute:
    def countKDifference(self, nums: List[int], k: int) -> int:
        n = len(nums)
        return sum(1 for i in range(n) for j in range(i + 1, n) if abs(nums[i] - nums[j]) == k)

print(SolutionBrute().countKDifference([3, 2, 1, 5, 4], 2))

3


Dòng `sum(1 for ... if ...)` là **generator expression**: sinh số `1` cho mỗi cặp thỏa điều kiện rồi cộng lại = đếm.

### Nút thắt của vét cạn

Với `n = 200` chỉ ~20 000 cặp — rất nhanh. Nhưng nếu `n = 10^5`: ~5 × 10⁹ cặp → quá chậm.

Quan sát: khi đứng ở `j` với giá trị `x = nums[j]`, các `i < j` hợp lệ là những chỉ số có `nums[i] == x − k` **hoặc** `nums[i] == x + k`. Ta không cần biết *vị trí* các `i` đó — chỉ cần **đếm** chúng.

In [8]:
for n in [200, 10_000, 100_000]:
    print(f"n = {n:>7}: số cặp = {n * (n - 1) // 2:,}")

n =     200: số cặp = 19,900
n =   10000: số cặp = 49,995,000
n =  100000: số cặp = 4,999,950,000


## 4. Insight — "Phần còn thiếu" có **hai** khả năng

`|a − x| = k` ⟺ `a − x = k` hoặc `a − x = −k` ⟺ `a = x + k` hoặc `a = x − k`.

Vì `k ≥ 1`, hai giá trị `x − k` và `x + k` **luôn khác nhau** → không lo đếm trùng giữa hai trường hợp.

Ví dụ `x = 3`, `k = 2`: cần tìm các số `1` hoặc `5` đã gặp trước đó.

In [9]:
x, k = 3, 2
need = [x - k, x + k]
print(f"x = {x}, k = {k} -> cần tìm số {need[0]} hoặc {need[1]} đã gặp")
for a in [1, 5, 3]:
    print(f"  |{a} - {x}| = {abs(a - x)} ->", "khớp" if abs(a - x) == k else "không")

x = 3, k = 2 -> cần tìm số 1 hoặc 5 đã gặp
  |1 - 3| = 2 -> khớp
  |5 - 3| = 2 -> khớp
  |3 - 3| = 0 -> không


## 5. Cách 2 — Hash map một lần duyệt (lời giải chính)

Giảng từng đoạn (giống Two Sum, nhưng **đếm** thay vì trả chỉ số):

```python
seen = {}                                   # giá trị -> số lần đã gặp (ở các vị trí trước j)
count = 0
for x in nums:                              # x = nums[j]
    count += seen.get(x - k, 0)             # (1) cặp với các i có nums[i] = x - k
    count += seen.get(x + k, 0)             # (2) cặp với các i có nums[i] = x + k
    seen[x] = seen.get(x, 0) + 1            # (3) SAU khi đếm mới thêm x
return count
```

**Kiểm tra trước, lưu sau:** đảm bảo chỉ ghép `x` với các phần tử **đứng trước** nó (`i < j`), mỗi cặp được đếm đúng một lần — tại thời điểm phần tử **sau** của cặp được duyệt.

In [10]:
def count_k_diff_hash(nums, k):
    seen = {}
    count = 0
    for x in nums:
        count += seen.get(x - k, 0) + seen.get(x + k, 0)
        seen[x] = seen.get(x, 0) + 1
    return count

print(count_k_diff_hash([1, 2, 2, 1], 1))
print(count_k_diff_hash([1, 3], 3))
print(count_k_diff_hash([3, 2, 1, 5, 4], 2))

4
0
3


## 6. TRACE từng bước (Cách 2)

Ví dụ `nums = [3, 2, 1, 5, 4]`, `k = 2`.

| j | x | seen **trước** | x−k → số lần | x+k → số lần | count sau | seen sau |
|:-:|:-:|---|---|---|:-:|---|
| 0 | 3 | {} | 1 → 0 | 5 → 0 | 0 | {3:1} |
| 1 | 2 | {3:1} | 0 → 0 | 4 → 0 | 0 | {3:1, 2:1} |
| 2 | 1 | {3:1, 2:1} | −1 → 0 | 3 → **1** | 1 | {3:1, 2:1, 1:1} |
| 3 | 5 | {3:1, 2:1, 1:1} | 3 → **1** | 7 → 0 | 2 | {…, 5:1} |
| 4 | 4 | {3:1, 2:1, 1:1, 5:1} | 2 → **1** | 6 → 0 | **3** | {…, 4:1} |

In [11]:
def trace_count_k_diff(nums, k):
    seen = {}
    count = 0
    print(f"nums = {nums}, k = {k}")
    for j, x in enumerate(nums):
        lo, hi = seen.get(x - k, 0), seen.get(x + k, 0)
        before = dict(seen)
        count += lo + hi
        seen[x] = seen.get(x, 0) + 1
        print(f"  j={j}, x={x} | seen trước = {before} | {x-k}->{lo}, {x+k}->{hi} | count = {count}")
    print("Kết quả:", count)
    return count

trace_count_k_diff([3, 2, 1, 5, 4], 2)
print()
trace_count_k_diff([1, 2, 2, 1], 1)

nums = [3, 2, 1, 5, 4], k = 2
  j=0, x=3 | seen trước = {} | 1->0, 5->0 | count = 0
  j=1, x=2 | seen trước = {3: 1} | 0->0, 4->0 | count = 0
  j=2, x=1 | seen trước = {3: 1, 2: 1} | -1->0, 3->1 | count = 1
  j=3, x=5 | seen trước = {3: 1, 2: 1, 1: 1} | 3->1, 7->0 | count = 2
  j=4, x=4 | seen trước = {3: 1, 2: 1, 1: 1, 5: 1} | 2->1, 6->0 | count = 3
Kết quả: 3

nums = [1, 2, 2, 1], k = 1
  j=0, x=1 | seen trước = {} | 0->0, 2->0 | count = 0
  j=1, x=2 | seen trước = {1: 1} | 1->1, 3->0 | count = 1
  j=2, x=2 | seen trước = {1: 1, 2: 1} | 1->1, 3->0 | count = 2
  j=3, x=1 | seen trước = {1: 1, 2: 2} | 0->0, 2->2 | count = 4
Kết quả: 4


4

Ở ví dụ `[1, 2, 2, 1]`: tại `j = 3` (x = 1), `seen = {1:1, 2:2}` → `x + k = 2` xuất hiện **2 lần** → cộng 2 cùng lúc. Đây là sức mạnh của **đếm**: một lần tra cứu thay cho nhiều cặp.

## 7. Code nộp LeetCode

In [12]:
class Solution:
    def countKDifference(self, nums: List[int], k: int) -> int:
        seen = Counter()
        count = 0
        for x in nums:
            count += seen[x - k] + seen[x + k]
            seen[x] += 1
        return count

for nums, k, expected in [([1, 2, 2, 1], 1, 4), ([1, 3], 3, 0), ([3, 2, 1, 5, 4], 2, 3)]:
    got = Solution().countKDifference(nums, k)
    print(nums, k, "->", got)
    assert got == expected

[1, 2, 2, 1] 1 -> 4
[1, 3] 3 -> 0
[3, 2, 1, 5, 4] 2 -> 3


Dùng `Counter` thay `dict` cho gọn: `seen[x - k]` trả 0 nếu chưa có.

*Lưu ý nhỏ:* tra `seen[key]` trên `Counter` với key chưa có **không** thêm key vào Counter (khác `defaultdict`). Kiểm chứng:

In [13]:
c = Counter()
_ = c[42]
print("Counter sau khi tra key 42:", c, "| 42 in c:", 42 in c)

Counter sau khi tra key 42: Counter() | 42 in c: False


## 8. Cách 3 — Đếm theo giá trị: `cnt[v] * cnt[v + k]`

Một góc nhìn khác: gom các phần tử theo **giá trị**. Với mỗi giá trị `v`, mọi phần tử bằng `v` ghép được với mọi phần tử bằng `v + k` → có `cnt[v] × cnt[v + k]` cặp chỉ số.

Chỉ xét `v + k` (không xét `v − k`) để mỗi cặp giá trị `{v, v+k}` được đếm **một lần**. Thứ tự `i < j` tự đúng: mỗi cặp vị trí khác nhau `{p, q}` ứng với đúng một cặp `(i, j)` có `i < j`.

Ví dụ `[1, 2, 2, 1]`, `k = 1`: `cnt = {1: 2, 2: 2}` → `cnt[1] × cnt[2] = 2 × 2 = 4`.

In [14]:
def count_k_diff_counter(nums, k):
    cnt = Counter(nums)
    return sum(c * cnt[v + k] for v, c in cnt.items())

print(count_k_diff_counter([1, 2, 2, 1], 1))
print(count_k_diff_counter([1, 3], 3))
print(count_k_diff_counter([3, 2, 1, 5, 4], 2))

4
0
3


*Cẩn thận:* trong `sum(...)` ta tra `cnt[v + k]` khi đang duyệt `cnt.items()`. Vì tra key chưa có **không** thêm key vào Counter (xem trên) nên không gặp lỗi "dictionary changed size during iteration". Nếu dùng `defaultdict` thì sẽ lỗi.

## 9. Cách 4 — Mảng đếm (giá trị nhỏ `1..100`)

Vì `1 <= nums[i] <= 100`, dùng list `freq` cỡ 101 thay cho hash map. Duyệt `v` từ 1 tới `100 − k`.

In [15]:
def count_k_diff_array(nums, k):
    freq = [0] * 101
    for x in nums:
        freq[x] += 1
    return sum(freq[v] * freq[v + k] for v in range(1, 101 - k))

print(count_k_diff_array([1, 2, 2, 1], 1))
print(count_k_diff_array([3, 2, 1, 5, 4], 2))
print(count_k_diff_array([1, 100], 99))

4
3
1


`range(1, 101 - k)` cho `v` từ 1 đến `100 − k`, đảm bảo `v + k ≤ 100` (không vượt chỉ số list).

## 10. Vì sao đúng?

**Cách 2 — Invariant:** ngay trước khi xử lý vị trí `j`, `seen[v]` = số chỉ số `i < j` có `nums[i] = v`; và `count` = số cặp hợp lệ `(i, j')` với `j' < j`.

- Tại `j`, các cặp mới `(i, j)` hợp lệ ⟺ `nums[i] ∈ {x − k, x + k}` với `i < j` → đúng là `seen[x−k] + seen[x+k]` (hai giá trị khác nhau vì `k ≥ 1`, nên không đếm trùng).
- Sau đó thêm `x` vào `seen` → invariant đúng cho `j + 1`.

Mỗi cặp `(i, j)` được đếm đúng một lần — khi duyệt tới `j` (phần tử sau).

**Cách 3:** mỗi cặp chỉ số thỏa có hai giá trị `{v, v + k}` với `v` là giá trị nhỏ hơn — duy nhất. Số cặp chỉ số có giá trị `{v, v+k}` là `cnt[v] · cnt[v+k]`. Cộng trên mọi `v` → đủ, không trùng.

## 11. Time / Space và bảng so sánh

Gọi `n = len(nums)`, `U` = số giá trị phân biệt, `V = 100`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Vét cạn | mọi cặp `i < j` | `O(n²)` | `O(1)` | Hint của đề, đủ nhanh với n ≤ 200 |
| 2. Hash map 1 lần duyệt | tra `x−k`, `x+k` rồi lưu | `O(n)` trung bình | `O(U)` | **Khuyến nghị**, tổng quát |
| 3. Counter theo giá trị | `cnt[v]·cnt[v+k]` | `O(n)` trung bình | `O(U)` | Gọn, đếm theo nhóm |
| 4. Mảng đếm | list cỡ 101 | `O(n + V)` | `O(V)` | Khi miền giá trị nhỏ |

"Trung bình" vì tra cứu hash map `O(1)` trung bình (không phải tuyệt đối mọi trường hợp).

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

1. Ba ví dụ đề.
2. Một phần tử → 0.
3. Toàn phần tử bằng nhau → 0 (vì `k ≥ 1`).
4. Cặp biên `[1, 100]`, `k = 99`.
5. Nhiều trùng lặp: `[1,1,1,2,2]`, `k=1` → 3 × 2 = 6.
6. Đối chiếu ngẫu nhiên giữa 4 cách.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS ({actual})")

f = count_k_diff_hash
check("đề 1", f([1, 2, 2, 1], 1), 4)
check("đề 2", f([1, 3], 3), 0)
check("đề 3", f([3, 2, 1, 5, 4], 2), 3)
check("một phần tử", f([7], 1), 0)
check("toàn bằng nhau", f([5, 5, 5], 1), 0)
check("biên [1, 100], k=99", f([1, 100], 99), 1)
check("nhiều trùng lặp", f([1, 1, 1, 2, 2], 1), 6)
check("x-k và x+k cùng có", f([1, 3, 5], 2), 2)

đề 1                       PASS (4)
đề 2                       PASS (0)
đề 3                       PASS (3)
một phần tử                PASS (0)
toàn bằng nhau             PASS (0)
biên [1, 100], k=99        PASS (1)
nhiều trùng lặp            PASS (6)
x-k và x+k cùng có         PASS (2)


In [17]:
import random

rng = random.Random(2006)
for _ in range(3000):
    nums = [rng.randint(1, 10) for _ in range(rng.randint(1, 20))]
    k = rng.randint(1, 9)
    exp = count_k_diff_brute(nums, k)
    assert count_k_diff_hash(nums, k) == exp
    assert count_k_diff_counter(nums, k) == exp
    assert count_k_diff_array(nums, k) == exp
    assert Solution().countKDifference(nums, k) == exp
print("3000 test ngẫu nhiên: 4 cách khớp với vét cạn")

3000 test ngẫu nhiên: 4 cách khớp với vét cạn


## 13. Lỗi người mới hay gặp

### Lỗi 1: Chỉ tra một phía `x − k` (quên `x + k`) trong Cách 2

In [18]:
def wrong_one_side(nums, k):
    seen, count = Counter(), 0
    for x in nums:
        count += seen[x - k]          # thiếu seen[x + k]
        seen[x] += 1
    return count

print("sai :", wrong_one_side([3, 2, 1, 5, 4], 2))   # 2
print("đúng:", count_k_diff_hash([3, 2, 1, 5, 4], 2))

sai : 2
đúng: 3


**Vì sao sai:** chỉ đếm được cặp mà phần tử **trước nhỏ hơn**. Cặp (3, 1) có phần tử trước lớn hơn bị bỏ sót. **Sửa:** tra cả `x − k` và `x + k`. (Cách 3 chỉ xét `v + k` là đúng **vì** nó duyệt theo giá trị, không theo vị trí.)

### Lỗi 2: Thêm `x` vào `seen` **trước** khi tra

In [19]:
def wrong_add_first(nums, k):
    seen, count = Counter(), 0
    for x in nums:
        seen[x] += 1
        count += seen[x - k] + seen[x + k]
    return count

print("kết quả:", wrong_add_first([1, 2, 2, 1], 1), "| đúng:", count_k_diff_hash([1, 2, 2, 1], 1))

kết quả: 4 | đúng: 4


Ở bài này kết quả **vẫn đúng** vì `k ≥ 1` nên `x ± k ≠ x` — thêm `x` sớm không ảnh hưởng. Nhưng thói quen này **sai** với biến thể `k = 0` (LeetCode 532 có `k = 0`): khi đó `x` tự ghép với chính nó.

In [20]:
def count_pairs_equal_wrong(nums):       # k = 0, thêm trước tra sau
    seen, count = Counter(), 0
    for x in nums:
        seen[x] += 1
        count += seen[x]
    return count

def count_pairs_equal_right(nums):
    seen, count = Counter(), 0
    for x in nums:
        count += seen[x]
        seen[x] += 1
    return count

print("k=0, [1,1]: sai =", count_pairs_equal_wrong([1, 1]), "| đúng =", count_pairs_equal_right([1, 1]))

k=0, [1,1]: sai = 3 | đúng = 1


**Sửa:** luôn **tra trước, lưu sau**.

### Lỗi 3: Cách 3 cộng cả `cnt[v−k]` và `cnt[v+k]` → đếm gấp đôi

In [21]:
def wrong_double(nums, k):
    cnt = Counter(nums)
    return sum(c * (cnt[v - k] + cnt[v + k]) for v, c in cnt.items())

print("sai :", wrong_double([1, 2, 2, 1], 1))   # 8
print("đúng:", count_k_diff_counter([1, 2, 2, 1], 1))

sai : 8
đúng: 4


**Vì sao sai:** cặp giá trị `{1, 2}` được đếm lúc `v = 1` (nhìn lên 2) **và** lúc `v = 2` (nhìn xuống 1). **Sửa:** chỉ nhìn một hướng, hoặc chia đôi kết quả.

### Lỗi 4: Đếm cặp **giá trị** phân biệt thay vì cặp **chỉ số**
Dùng `set` các cặp giá trị → ví dụ 1 ra 1 thay vì 4. Đề đếm `(i, j)`, không loại trùng giá trị.

## 14. Mẹo nhận dạng pattern

- "Đếm số cặp `i < j` thỏa quan hệ giữa `nums[i]` và `nums[j]`" → duyệt `j`, **hash map đếm** các giá trị đã gặp, tra "phần còn thiếu".
- Có **giá trị tuyệt đối** → phần còn thiếu có **hai** khả năng (`x − k`, `x + k`).
- Kết quả chỉ phụ thuộc giá trị (không phụ thuộc vị trí ngoài `i < j`) → có thể **gom theo giá trị** và nhân số lần.
- Giá trị nhỏ, bị chặn → **mảng đếm**.

## 15. Biến thể

1. **LeetCode 1512 — Number of Good Pairs:** `nums[i] == nums[j]` (`k = 0`): `count += seen[x]` rồi `seen[x] += 1`.
2. **LeetCode 532 — K-diff Pairs:** đếm cặp **giá trị phân biệt** (không theo chỉ số), có `k = 0`.
3. **Đếm cặp có hiệu ≤ k:** sort + hai con trỏ.

In [22]:
def good_pairs(nums):                    # LC 1512
    seen, count = Counter(), 0
    for x in nums:
        count += seen[x]
        seen[x] += 1
    return count

def k_diff_unique_pairs(nums, k):        # LC 532
    cnt = Counter(nums)
    if k == 0:
        return sum(1 for c in cnt.values() if c >= 2)
    return sum(1 for v in cnt if v + k in cnt)

print("LC 1512 [1,2,3,1,1,3]:", good_pairs([1, 2, 3, 1, 1, 3]))
print("LC 532 [3,1,4,1,5], k=2:", k_diff_unique_pairs([3, 1, 4, 1, 5], 2))
print("LC 532 [1,3,1,5,4], k=0:", k_diff_unique_pairs([1, 3, 1, 5, 4], 0))

LC 1512 [1,2,3,1,1,3]: 4
LC 532 [3,1,4,1,5], k=2: 2
LC 532 [1,3,1,5,4], k=0: 1


## 16. Bài tập tự luyện (TODO — chưa chạy)

### Bài 1 — Tính tay
Lập bảng trace Cách 2 cho `nums = [1, 3, 5, 3]`, `k = 2`. Đáp án?

### Bài 2 — Dự đoán output
`count_k_diff_counter([2, 2, 4, 4, 4], 2)` = ? Giải thích bằng phép nhân.

### Bài 3 — Tự viết lời giải hash map

In [23]:
def exercise_count_k_diff(nums: list[int], k: int) -> int:
    # TODO 1: seen = Counter(), count = 0
    # TODO 2: với mỗi x: count += seen[x - k] + seen[x + k]
    # TODO 3: seen[x] += 1 (SAU khi tra)
    pass

# assert exercise_count_k_diff([1, 2, 2, 1], 1) == 4
# assert exercise_count_k_diff([3, 2, 1, 5, 4], 2) == 3

### Bài 4 — Biến thể: LeetCode 1512 — Good Pairs

In [24]:
def exercise_good_pairs(nums: list[int]) -> int:
    # TODO: đếm cặp i < j có nums[i] == nums[j]
    pass

# assert exercise_good_pairs([1, 2, 3, 1, 1, 3]) == 4
# assert exercise_good_pairs([1, 1, 1, 1]) == 6

### Bài 5 — Biến thể: đếm cặp `i < j` có `nums[i] + nums[j] == target`

In [25]:
def exercise_count_pair_sum(nums: list[int], target: int) -> int:
    # TODO: phần còn thiếu là target - x
    pass

# assert exercise_count_pair_sum([1, 5, 7, -1, 5], 6) == 3
# assert exercise_count_pair_sum([1, 1, 1, 1], 2) == 6

### Bài 6 — Mini challenge: đếm cặp `i < j` có `|nums[i] − nums[j]| <= k` trong `O(n log n)`

In [26]:
def exercise_count_within(nums: list[int], k: int) -> int:
    # TODO: sort; hai con trỏ left/right; với mỗi right, dịch left tới khi a[right] - a[left] <= k
    # cộng (right - left) cặp
    pass

# assert exercise_count_within([1, 3, 6, 7], 2) == 2   # (1,3), (6,7)
# assert exercise_count_within([5, 5, 5], 0) == 3

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Tra `x−2`, `x+2` mỗi bước | j=1 (x=3): seen có 1 | tổng 0 + 1 + 1 + 1 |
| 2 | `cnt = {2: 2, 4: 3}` | chỉ `v = 2` có `v+2` trong cnt | `2 × 3` |
| 3 | Khuôn mục 5 | `Counter` | tra trước, lưu sau |
| 4 | Phần còn thiếu là chính `x` | `count += seen[x]` | rồi `seen[x] += 1` |
| 5 | Như Two Sum nhưng đếm | `count += seen[target - x]` | rồi `seen[x] += 1` |
| 6 | Sort trước | cửa sổ co giãn | `count += right - left` |

## 18. Đáp án tham khảo, template và checklist

- Bài 1: j=0 (1): 0 → seen {1:1}; j=1 (3): seen[1]=1 → 1; j=2 (5): seen[3]=1 → 2; j=3 (3): seen[1]=1, seen[5]=1 → **4**.
- Bài 2: `cnt = {2:2, 4:3}` → `cnt[2] × cnt[4] = 2 × 3 =` **6**; `v = 4` nhìn lên 6 → 0.

In [27]:
def answer_count_k_diff(nums, k):
    seen, count = Counter(), 0
    for x in nums:
        count += seen[x - k] + seen[x + k]
        seen[x] += 1
    return count

def answer_good_pairs(nums):
    seen, count = Counter(), 0
    for x in nums:
        count += seen[x]
        seen[x] += 1
    return count

def answer_count_pair_sum(nums, target):
    seen, count = Counter(), 0
    for x in nums:
        count += seen[target - x]
        seen[x] += 1
    return count

def answer_count_within(nums, k):
    a = sorted(nums)
    count = left = 0
    for right in range(len(a)):
        while a[right] - a[left] > k:
            left += 1
        count += right - left
    return count

assert answer_count_k_diff([1, 3, 5, 3], 2) == 4
assert count_k_diff_counter([2, 2, 4, 4, 4], 2) == 6
assert answer_count_k_diff([1, 2, 2, 1], 1) == 4 and answer_count_k_diff([3, 2, 1, 5, 4], 2) == 3
assert answer_good_pairs([1, 2, 3, 1, 1, 3]) == 4 and answer_good_pairs([1, 1, 1, 1]) == 6
assert answer_count_pair_sum([1, 5, 7, -1, 5], 6) == 3 and answer_count_pair_sum([1, 1, 1, 1], 2) == 6
assert answer_count_within([1, 3, 6, 7], 2) == 2 and answer_count_within([5, 5, 5], 0) == 3
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def countKDifference(self, nums: List[int], k: int) -> int:
        seen, count = Counter(), 0
        for x in nums:
            count += seen[x - k] + seen[x + k]   # tra hai "phần còn thiếu"
            seen[x] += 1                          # lưu sau
        return count
```

Một dòng: `cnt = Counter(nums); return sum(c * cnt[v + k] for v, c in cnt.items())`.

### Checklist trước khi submit
- [ ] Đếm theo **chỉ số** (`i < j`), không loại trùng giá trị?
- [ ] Tra **cả** `x − k` và `x + k` (nếu duyệt theo vị trí)?
- [ ] Nếu đếm theo giá trị: chỉ nhìn **một hướng** (`v + k`)?
- [ ] Tra trước, lưu sau (an toàn cả khi `k = 0`)?
- [ ] Đã test mảng có nhiều phần tử trùng (`[1,2,2,1]` → 4)?